In [ ]:
!git clone --depth 1 https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN

In [ ]:
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt

In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime2/cm_4.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
QUALITY_PRESET = "safe"       # baseline / safe
SCALE = 2
FPS = "source"

START_TIME = 3 * 60 + 15
TEST_SECONDS = 10
RUN_BOTH_10S_TESTS = True         # 10 秒时依次生成 baseline 与 safe
PROGRESS_INTERVAL = 60.0

NATIVE_ANALYSIS = "off"
NATIVE_SAMPLES = 5
NATIVE_MIN_HEIGHT = 500
NATIVE_MAX_HEIGHT = 1080
NATIVE_KERNELS = "bilinear,bicubic,lanczos"
NATIVE_CONFIDENCE = 0.85
NATIVE_HEIGHT = 0
NATIVE_KERNEL = "auto"
DESCALE = False

INPUT_WIDTH = 0
INPUT_HEIGHT = 0
TILE_SIZE = 0                    # OOM 时改 576，再改 256
TILE_PAD = 10                    # 每个 tile 的模型上下文
PRE_PAD = 0                      # 整帧外边界全局 padding，与 TILE_PAD 独立
TILE_VERIFY_COVERAGE = True
BATCH_SIZE = 8
GPU_IDS = "0,1"
TTA_BATCH_SIZE = 1

BACK_PROJECTION_STRENGTH = 0.2
BACK_PROJECTION_KERNEL = "lanczos"
BACK_PROJECTION_CLAMP = 0.05
DEHALO_RADIUS = 2
RANGE_RADIUS = 2
OVERSHOOT = 1.0
UNDERSHOOT = 1.0

ANIME4K = False
ANIME4K_SHADER_DIR = ""
ANIME4K_SHADERS = ""
ANIME4K_STRENGTH = 1.0

VIDEO_CODEC = "hevc_nvenc"
OUTPUT_PIX_FMT = "auto"
CRF = 18
PRESET = "medium"
CQ = 18
NVENC_PRESET = "p7"
ENCODE_GPU = 0
AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"

# 高级覆盖项会覆盖预设；默认留空以避免预设/显式参数冲突。
EXTRA_ARGS = []

In [ ]:
import shlex
import subprocess
import sys
from pathlib import Path

presets = ["baseline", "safe"] if RUN_BOTH_10S_TESTS and TEST_SECONDS == 10 else [QUALITY_PRESET]
for quality_preset in presets:
    output = Path(OUTPUT_VIDEO)
    if len(presets) > 1:
        output = output.with_name(f"{output.stem}_{quality_preset}{output.suffix}")
    command = [
        sys.executable, "/kaggle/working/Real-ESRGAN/realesrgan.py",
        "--input", INPUT_VIDEO, "--output", str(output),
        "--model", MODEL, "--model-path", MODEL_PATH,
        "--quality-preset", quality_preset, "--scale", str(SCALE), "--fps", FPS,
        "--fp16", "--channels-last",
        "--native-analysis", NATIVE_ANALYSIS, "--native-samples", str(NATIVE_SAMPLES),
        "--native-min-height", str(NATIVE_MIN_HEIGHT), "--native-max-height", str(NATIVE_MAX_HEIGHT),
        "--native-kernels", NATIVE_KERNELS, "--native-confidence", str(NATIVE_CONFIDENCE),
        "--native-height", str(NATIVE_HEIGHT), "--native-kernel", NATIVE_KERNEL,
        "--descale" if DESCALE else "--no-descale",
        "--input-width", str(INPUT_WIDTH), "--input-height", str(INPUT_HEIGHT),
        "--tile-size", str(TILE_SIZE), "--tile-pad", str(TILE_PAD), "--pre-pad", str(PRE_PAD),
        "--tile-verify-coverage" if TILE_VERIFY_COVERAGE else "--no-tile-verify-coverage",
        "--batch-size", str(BATCH_SIZE), "--gpu-ids", GPU_IDS,
        "--tta-batch-size", str(TTA_BATCH_SIZE),
        "--back-projection-strength", str(BACK_PROJECTION_STRENGTH),
        "--back-projection-kernel", BACK_PROJECTION_KERNEL,
        "--back-projection-clamp", str(BACK_PROJECTION_CLAMP),
        "--dehalo-radius", str(DEHALO_RADIUS), "--range-radius", str(RANGE_RADIUS),
        "--overshoot", str(OVERSHOOT), "--undershoot", str(UNDERSHOOT),
        "--anime4k" if ANIME4K else "--no-anime4k",
        "--anime4k-shader-dir", ANIME4K_SHADER_DIR,
        "--anime4k-shaders", ANIME4K_SHADERS, "--anime4k-strength", str(ANIME4K_STRENGTH),
        "--video-codec", VIDEO_CODEC, "--output-pix-fmt", OUTPUT_PIX_FMT,
        "--crf", str(CRF), "--preset", PRESET, "--cq", str(CQ),
        "--nvenc-preset", NVENC_PRESET, "--encode-gpu", str(ENCODE_GPU),
        "--audio-codec", AUDIO_CODEC, "--audio-bitrate", AUDIO_BITRATE,
        "--start-time", str(START_TIME), "--test-seconds", str(TEST_SECONDS),
        "--progress-interval", str(PROGRESS_INTERVAL), "--ffmpeg-bin", "ffmpeg",
        "--ffprobe-bin", "ffprobe", *EXTRA_ARGS,
    ]
    print("[command]", shlex.join(command), flush=True)
    subprocess.run(command, check=True)

## 可选能力（仅在需要时运行）

Descale/getnative 与 Anime4K 不属于 baseline/safe 必需项。显式启用时必须通过真实能力检查。

In [ ]:
!pip install -q wrapt VapourSynth==77 vapoursynth-descale==12 vapoursynth-ffms2==5.2.1 getnative==3.3.0 || echo "可选 Descale 依赖安装失败"
!vapoursynth config || true
!vapoursynth check-env || true
!command -v vspipe >/dev/null && getnative --help >/dev/null && python -c "import vapoursynth as vs; assert hasattr(vs.core,'descale') and hasattr(vs.core,'ffms2')" && echo "getnative/Descale: available" || echo "Descale 能力不完整，不要启用 DESCALE"

In [ ]:
!ffmpeg -hide_banner -filters 2>&1 | grep -q libplacebo && echo "FFmpeg libplacebo: available" || echo "FFmpeg libplacebo: unavailable；保持 ANIME4K=False"